In [ ]:
# =========================================================
# 1. SETUP (ANTI CUDA ERROR)
# =========================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# =========================================================
# 2. IMPORT LIBRARY
# =========================================================
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================================================
# 3. DATASET PATH & CLASS MAP
# =========================================================
original_dataset_path = '/kaggle/input/waste-classification'
custom_dataset_path = '/kaggle/working/final_dataset'

class_map = {
    'Organic': 'biodegradable',
    'Recyclable': 'recyclable_plastic',
    'Non-Recyclable' : 'non_recyclable',
    'Hazardous': 'non_recyclable'
}

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

# =========================================================
# 4. BUILD TRAIN / VAL DATASET
# =========================================================
if os.path.exists(custom_dataset_path):
    shutil.rmtree(custom_dataset_path)

for split in ['train', 'val']:
    for cls in class_map.values():
        os.makedirs(os.path.join(custom_dataset_path, split, cls), exist_ok=True)

for main_folder, target_class in class_map.items():
    level2_path = os.path.join(original_dataset_path, main_folder, main_folder)
    if not os.path.isdir(level2_path):
        continue
    for category in os.listdir(level2_path):
        category_path = os.path.join(level2_path, category)
        if not os.path.isdir(category_path):
            continue
        images = [
            f for f in os.listdir(category_path)
            if os.path.isfile(os.path.join(category_path, f)) and f.lower().endswith(IMAGE_EXTENSIONS)
        ]
        if len(images) < 2:
            continue
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
        for img in train_imgs:
            shutil.copy(
                os.path.join(category_path, img),
                os.path.join(custom_dataset_path, 'train', target_class, img)
            )
        for img in val_imgs:
            shutil.copy(
                os.path.join(category_path, img),
                os.path.join(custom_dataset_path, 'val', target_class, img)
            )

print("✅ Dataset siap")

# =========================================================
# 5. IMAGE DATA GENERATOR
# =========================================================
img_size = (224, 224)
batch_size = 32

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
).flow_from_directory(
    os.path.join(custom_dataset_path, 'train'),
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    os.path.join(custom_dataset_path, 'val'),
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("Class mapping:", train_gen.class_indices)

# =========================================================
# 6. CALLBACKS
# =========================================================
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

# =========================================================
# 7. OPTUNA (STAGE-1 SEARCH)
# =========================================================
def build_model(trial):
    lr = trial.suggest_float("lr", 1e-5, 3e-4, log=True)
    dropout = trial.suggest_float("dropout", 0.3, 0.6)
    dense_units = trial.suggest_int("dense_units", 128, 256, step=64)

    base_model = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_tensor=Input(shape=(224,224,3))
    )
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(dense_units, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)
    output = Dense(3, activation="softmax")(x)

    model = Model(base_model.input, output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model

def objective(trial):
    model = build_model(trial)
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=5,
        verbose=0
    )
    return max(history.history["val_accuracy"])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)
best_params = study.best_params
print("🏆 Best Params:", best_params)

# =========================================================
# 8. STAGE-1 TRAINING (BEST MODEL)
# =========================================================
base_model = EfficientNetB0(weights="imagenet", include_top=False, input_tensor=Input(shape=(224,224,3)))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(best_params["dense_units"], activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(best_params["dropout"])(x)
output = Dense(3, activation="softmax")(x)

model = Model(base_model.input, output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(best_params["lr"]),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks
)

# =========================================================
# 9. STAGE-2 FINE-TUNING
# =========================================================
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_stage2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)

# =========================================================
# 10. EVALUATION
# =========================================================
y_true = val_gen.classes
y_pred = np.argmax(model.predict(val_gen), axis=1)

print(classification_report(y_true, y_pred, target_names=val_gen.class_indices.keys()))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=val_gen.class_indices.keys(),
            yticklabels=val_gen.class_indices.keys())
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# =========================================================
# 11. PLOT HISTORY
# =========================================================
def plot_history(h1, h2):
    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(h1.history["accuracy"], label="Stage-1 Train")
    plt.plot(h1.history["val_accuracy"], label="Stage-1 Val")
    plt.plot(h2.history["accuracy"], label="Stage-2 Train")
    plt.plot(h2.history["val_accuracy"], label="Stage-2 Val")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1,2,2)
    plt.plot(h1.history["loss"], label="Stage-1 Train")
    plt.plot(h1.history["val_loss"], label="Stage-1 Val")
    plt.plot(h2.history["loss"], label="Stage-2 Train")
    plt.plot(h2.history["val_loss"], label="Stage-2 Val")
    plt.legend()
    plt.title("Loss")

    plt.show()

plot_history(history_stage1, history_stage2)

# =========================================================
# 12. SAVE MODEL
# =========================================================
model.save('/kaggle/working/waste_classifier.keras')
print("✅ MODEL FINAL SAVED – READY FOR DEPLOY")